### **Prerequisites for the Lab**

Before the session, ensure you have the environment set up.

1. Install [Ollama](https://ollama.com/) and download a lightweight model: `ollama run llama3` (or `llama3.1` or `mistral`).
2. Install the required Python libraries:

In [7]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,010 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [8]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [9]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

In [10]:
!ollama pull gemma4:latest

In [11]:
!pip install langchain langchain-ollama langchain-community

### 1. The Tool-Use Pattern (Enterprise Data Access)

In [12]:
from langchain_core.tools import tool

# 1. Define the Enterprise Tool
@tool
def fetch_ats_resume(candidate_name: str) -> str:
    """Searches the Applicant Tracking System (ATS) for a candidate's resume summary."""
    # Simulating a secure internal database lookup
    ats_database = {
        "alice smith": "4 years Data Science experience. Strong in Python and cloud architectures.",
        "bob jones": "2 years Frontend Web Development. Good communication."
    }
    return ats_database.get(candidate_name.lower(), "Candidate not found in ATS.")

In [13]:
# 2. Test the Tool Directly
print(" Testing Enterprise Tool-Use ")
result = fetch_ats_resume.invoke({"candidate_name": "Alice Smith"})
print(f"Secure Database Output: {result}")

 Testing Enterprise Tool-Use 
Secure Database Output: 4 years Data Science experience. Strong in Python and cloud architectures.


### 2. The ReAct Pattern (Reason + Act)

The agent thinks, decides to use a tool, observes the result, and formulates a final answer.

In [14]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
# Import from langchain_classic instead of langchain
from langchain_classic.agents import initialize_agent, AgentType

In [15]:
# 1. Initialize local model
llm = ChatOllama(model="gemma4:latest", temperature=0)

In [16]:
# 2. Define the tool
@tool
def check_job_requirements(job_id: str) -> str:
    """Fetches the required skills for a specific Job ID."""
    return "Job ID A100: Requires Python, enterprise frameworks, and 3+ years of Data Science experience."

In [17]:
# 3. Initialize the ReAct Agent
react_agent = initialize_agent(
    tools=[check_job_requirements],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True # Shows the Thought -> Action -> Observation loop
)

In [18]:
# 4. Execute
print("\n Running ReAct Agent ")
response = react_agent.run("What specific skills are required for Job ID A100?")
print(f"\nFinal Agent Answer: {response}")


 Running ReAct Agent 


> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "check_job_requirements",
  "action_input": {
    "job_id": "A100"
  }
}
```
Observation: Job ID A100: Requires Python, enterprise frameworks, and 3+ years of Data Science experience.
Thought:Action:
```json
{
  "action": "Final Answer",
  "action_input": "The specific skills required for Job ID A100 are Python, enterprise frameworks, and 3+ years of Data Science experience."
}
```

> Finished chain.

Final Agent Answer: The specific skills required for Job ID A100 are Python, enterprise frameworks, and 3+ years of Data Science experience.


### 3. The Planner Pattern (Task Decomposition)

Handle massive, ambiguous requests by breaking them down into a sequence of logical steps (preventing agent hallucination or infinite loops).

In [19]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

In [23]:
# 1. Initialize local model
llm = ChatOllama(model="gemma4:latest", temperature=0)

In [24]:
# 2. The Planner Prompt
planner_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an Enterprise Workflow Architect.
    Given a high-level goal, break it down into a strict, step-by-step JSON list of tasks.
    Do not execute the steps. Just create the strategic plan.
    Available Tools to consider: [Search_ATS, Extract_Resume, Draft_Email, Schedule_Meeting]"""),
    ("user", "Goal: {goal}")
])

planner_chain = planner_prompt | llm

In [25]:
# 3. Execute
print("\n Generating Execution Plan ")
complex_goal = "We need to hire 3 Senior Data Scientists. Find candidates, screen them against the job description, and set up initial calls."

plan = planner_chain.invoke({"goal": complex_goal}).content
print(plan)


 Generating Execution Plan 
```json
{
  "workflow_title": "Senior Data Scientist Hiring Pipeline",
  "goal": "Identify, screen, and schedule initial calls with 3 qualified Senior Data Scientists.",
  "prerequisites": [
    "Finalized Job Description (JD) for Senior Data Scientist role.",
    "Access credentials for the Applicant Tracking System (ATS)."
  ],
  "workflow_steps": [
    {
      "step_id": 1,
      "phase": "Sourcing and Candidate Identification",
      "task_description": "Utilize the ATS to search for candidates matching core technical skills (e.g., Python, ML frameworks, specific industry experience) defined in the JD.",
      "tools_required": [
        "Search_ATS"
      ],
      "output": "A prioritized list of 20-30 potential candidate profiles."
    },
    {
      "step_id": 2,
      "phase": "Deep Screening and Qualification",
      "task_description": "For the top 15 candidates identified in Step 1, extract their full resumes. Systematically screen each resume ag

### 4. The Executor Pattern (Focused Execution Engine)

Executor takes a single, isolated step from the Planner and uses wikipedia tools to accomplish exactly that one task without getting distracted.

In [50]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=b8bff9fe345dfa106df3294211bb6252c778f665eb1cc7787937372b21aa6df6
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [56]:
import urllib.request
import urllib.parse
import json
from langchain_core.tools import tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_ollama import ChatOllama

In [57]:
# 1. Define a robust custom Wikipedia tool that handles bad model parsing cleanly
@tool
def robust_wikipedia_search(query: str | dict) -> str:
    """Searches Wikipedia and extracts a clean text summary of a concept."""
    # Handle gemma4 outputting complex dict variants smoothly
    if isinstance(query, dict):
        search_term = query.get("query", query.get("title", str(query)))
    else:
        search_term = query

    try:
        # Step A: Query MediaWiki API to search for the closest matching page title
        search_url = f"https://en.wikipedia.org/w/api.php?action=opensearch&search={urllib.parse.quote(search_term)}&limit=1&namespace=0&format=json"
        req = urllib.request.Request(search_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())

        if not data[1]:
            return f"No direct Wikipedia matches found for search term: '{search_term}'."

        exact_title = data[1][0]

        # Step B: Fetch the plain-text introduction extract of that exact page
        extract_url = f"https://en.wikipedia.org/w/api.php?action=query&prop=extracts&exintro=1&explaintext=1&titles={urllib.parse.quote(exact_title)}&format=json"
        req_extract = urllib.request.Request(extract_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req_extract) as extract_response:
            extract_data = json.loads(extract_response.read().decode())

        pages = extract_data['query']['pages']
        page_id = list(pages.keys())[0]

        if page_id == "-1":
            return f"Could not retrieve summary for '{exact_title}'."

        return f"Wikipedia Summary for [{exact_title}]:\n{pages[page_id]['extract']}"

    except Exception as e:
        return f"An error occurred while connecting to Wikipedia: {str(e)}"

In [58]:
# 2. Re-initialize your local model
llm = ChatOllama(model="gemma4:latest", temperature=0)

In [59]:
# 3. Build the Executor Agent using our resilient custom tool
executor_agent = initialize_agent(
    tools=[robust_wikipedia_search],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [60]:
# 4. Run the live search task with a clear nudge to formulate the Final Answer
print("\n Running the Executor \n")
assigned_task = "Search Wikipedia for 'Data Science' and provide a brief summary of what it is. Make sure to conclude with a clear Final Answer."
executor_result = executor_agent.run(assigned_task)

print(f"\nExecutor Output:\n{executor_result}")


 Running the Executor 



> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "robust_wikipedia_search",
  "action_input": {
    "query": "Data Science",
    "title": "Data Science"
  }
}
```
Observation: Wikipedia Summary for [Data science]:
Data science is an interdisciplinary academic field that uses statistics, scientific computing, scientific methods, processing, scientific visualization, algorithms, coding (like Python, SQL, and R), and systems to extract or extrapolate knowledge from potentially noisy, structured, or unstructured data. A data scientist is a professional who creates programming code and combines it with statistical knowledge to summarize data.
Thought:Action:
```json
{
  "action": "Final Answer",
  "action_input": "Data science is an interdisciplinary academic field that uses statistics, scientific computing, scientific methods, processing, scientific visualization, algorithms, and coding (such as Python, SQL, and R) to extract or extrapolate kno

### 5. The Evaluator / Reflection Pattern (Quality & Compliance)

The Evaluator acts as an automated firewall, reviewing generated content against corporate guidelines before human review.

In [43]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

In [44]:
# 1. Initialize local model
llm = ChatOllama(model="gemma4:latest", temperature=0)

In [45]:
# 2. The raw output simulating a poorly drafted Executor response
raw_executor_output = """
The candidate Bob failed the coding test.
He only got a 40%. We are rejecting him immediately. Tell him he didn't make the cut.
"""

In [46]:
# 3. Build the Evaluator
evaluator_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are the Enterprise Evaluator and HR Compliance Officer.
    Review the Executor's output and ensure it meets corporate communication standards.

    Grading Rubric:
    1. Is the tone strictly professional and empathetic?
    2. Does it protect the company's employer brand?

    If it fails the rubric, rewrite the text to be highly professional and corporate-friendly."""),
    ("user", "Executor Output to Evaluate:\n{output}")
])

evaluator_chain = evaluator_prompt | llm

In [47]:
# 4. Execute
print("\n Running the Evaluator ")
print(f"Raw Input to Evaluate:\n{raw_executor_output}")


 Running the Evaluator 
Raw Input to Evaluate:

The candidate Bob failed the coding test. 
He only got a 40%. We are rejecting him immediately. Tell him he didn't make the cut.



In [48]:
print("\n Evaluator Analysis & Correction ")
evaluation_result = evaluator_chain.invoke({"output": raw_executor_output}).content
print(evaluation_result)


 Evaluator Analysis & Correction 
## Enterprise Evaluation Report

**Review Status:** Failed
**Compliance Risk Level:** High (Poor candidate experience, damages employer brand)

### Analysis Against Grading Rubric

**1. Is the tone strictly professional and empathetic?**
**Verdict: FAIL.** The language is overly blunt ("failed," "didn't make the cut") and lacks any acknowledgment of the candidate's effort or time invested in the process. This approach is dismissive and unprofessional.

**2. Does it protect the company's employer brand?**
**Verdict: FAIL.** Communicating rejection this way suggests that the company values harshness over professionalism. A poor candidate experience can lead to negative reviews, damaging our reputation as an employer of choice.

***

### HR Compliance & Corporate Rewrite

The original text is unsuitable for any corporate communication channel (internal instruction or external email). The language must be softened, depersonalized, and focused on objective